In [1]:
from selenium import webdriver  #4.25.0
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
import re
import csv

### Crawl sport venues

In [18]:
chrome_options = Options()
chrome_options.add_argument("--headless")
chrome_options.add_argument("--disable-gpu")
chrome_options.add_argument("--window-size=1920,1080")

driver = webdriver.Chrome(options=chrome_options)
wait = WebDriverWait(driver, 5)

venues_data = []
venue_links = []

try:
    category_url = "https://en.wikipedia.org/wiki/Category:Sports_venues_in_Vietnam"
    driver.get(category_url)
    
    # Fetch Subcategory Links
    try:
        mw_subcategories = wait.until(EC.presence_of_element_located((By.ID, 'mw-subcategories')))
        links = mw_subcategories.find_elements(By.TAG_NAME, "a")
        subcategory_urls = [link.get_attribute("href") for link in links if link.get_attribute("href")]
        print(f"There are {len(subcategory_urls)} subcategory links.")
    except TimeoutException:
        print("Could not find the subcategories section.")
        subcategory_urls = []

    # Fetch Venue Links from Subcategories
    for url in subcategory_urls:
        driver.get(url)
        try:
            mw_pages = wait.until(EC.presence_of_element_located((By.ID, 'mw-pages')))
            page_links = mw_pages.find_elements(By.TAG_NAME, "a")
            venue_links.extend([link.get_attribute("href") for link in page_links if link.get_attribute("href")])
        except TimeoutException:
            print(f"Could not find pages in subcategory: {url}")
            
    print(f"There are {len(venue_links)} total venue links.")

    # Scrape Venue Data
    for link in venue_links:
        driver.get(link)
        try:
            venue_name = wait.until(EC.presence_of_element_located((By.ID, 'firstHeading'))).text
            infobox_data = {}
            
            # Extract Infobox Data
            try:
                infobox = driver.find_element(By.CSS_SELECTOR, "table.infobox") 
                rows = infobox.find_elements(By.TAG_NAME, "tr")
                for row in rows:
                    try:
                        header = row.find_element(By.TAG_NAME, "th").text.strip()
                        value = row.find_element(By.TAG_NAME, "td").text.strip()
                        infobox_data[header] = value
                    except NoSuchElementException:
                        continue 
            except NoSuchElementException:
                pass
            
            # Filter and Append
            if re.search(r"Wikipedia|List of", venue_name, re.IGNORECASE):
                continue
            else:
                venues_data.append({
                    "name": venue_name,
                    'url': link,
                    "infobox": infobox_data
                })            
        except TimeoutException:
            print(f"Timeout loading venue page: {link}")

finally:
    driver.quit()

There are 8 subcategory links.
There are 64 total venue links.


### Preprocess data

In [ ]:
def clean_venue_data_with_city(patched_list):
    cleaned_venues = []
    
    for item in patched_list:
        name = item.get('name', '').strip()

        infobox = item.get('infobox', {})
        url = item.get('url', '').strip()


        raw_address = infobox.get('Address') or infobox.get('Location') or ''
        clean_address = re.sub(r'\[.*?\]', '', raw_address)
        clean_address = clean_address.replace('\n', ', ').strip()
        
        # Extract the City and Trim Address
        city = "-1" 
        final_street_address = clean_address
        
        if clean_address:
            address_parts = [part.strip() for part in clean_address.split(',') if part.strip()]
            
            if len(address_parts) > 0:
                if address_parts[-1].lower() == 'vietnam':
                    if len(address_parts) > 1:
                        city = address_parts[-2]
                        remaining_parts = address_parts[:-2]
                    else:
                        remaining_parts = [] # Only "Vietnam" was in the address
                else:
                    city = address_parts[-1]
                    remaining_parts = address_parts[:-1]
                
                if remaining_parts:
                    final_street_address = ", ".join(remaining_parts)
                else:
                    final_street_address = "-1"
        else:
            final_street_address = "-1"

        # Extract strictly integer Capacity
        raw_capacity = infobox.get('Capacity', '')
        clean_capacity = None
        if raw_capacity:
            match = re.search(r'[\d,.]+', raw_capacity)
            if match:
                num_str = match.group(0).replace(',', '').replace('.', '')
                try:
                    clean_capacity = int(num_str)
                except ValueError:
                    pass
                    
        # Append into the right format
        cleaned_venues.append({
            'name': name,
            'description': url,         
            'phone': -1,             
            'email': None,             
            'website': None,           
            'address_line': final_street_address, 
            'city': city,                    
            'capacity': clean_capacity
        })
        
    return cleaned_venues

In [20]:
final = clean_venue_data_with_city(venues_data)

In [27]:
final

[{'name': 'Hàng Đẫy Stadium',
  'description': 'https://en.wikipedia.org/wiki/H%C3%A0ng_%C4%90%E1%BA%ABy_Stadium',
  'phone': None,
  'email': None,
  'website': None,
  'address_line': 'Ô Chợ Dừa',
  'city': 'Hanoi',
  'capacity': 22500},
 {'name': 'Mỹ Đình National Stadium',
  'description': 'https://en.wikipedia.org/wiki/M%E1%BB%B9_%C4%90%C3%ACnh_National_Stadium',
  'phone': None,
  'email': None,
  'website': None,
  'address_line': '1 Lê Đức Thọ Rd, Từ Liêm Ward',
  'city': 'Hanoi',
  'capacity': 40192},
 {'name': 'Trống Đồng Stadium',
  'description': 'https://en.wikipedia.org/wiki/Tr%E1%BB%91ng_%C4%90%E1%BB%93ng_Stadium',
  'phone': None,
  'email': None,
  'website': None,
  'address_line': 'Thượng Phúc',
  'city': 'Hanoi',
  'capacity': 135000},
 {'name': 'Hanoi Indoor Games Gymnasium',
  'description': 'https://en.wikipedia.org/wiki/Hanoi_Indoor_Games_Gymnasium',
  'phone': None,
  'email': None,
  'website': None,
  'address_line': 'Trần Hữu Dực Street, Cầu Diễn Ward, Nam T

### Save to csv format

In [28]:
def export_to_csv(cleaned_data, filename="vietnam_sports_venues.csv"):
    if not cleaned_data:
        print("The dataset is empty.")
        return

    fieldnames = ['name', 'description','phone','email','website', 'address_line', 'city', 'capacity']

    try:
        with open(filename, mode='w', newline='', encoding='utf-8-sig') as csv_file:
            writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(cleaned_data)
            
        print(f"\n {len(cleaned_data)} venues saved to '{filename}'.")
    except IOError as e:
        print(f"An error: {e}")


In [29]:
export_to_csv(final)


 55 venues saved to 'vietnam_sports_venues.csv'.


In [24]:
import pandas as pd
from sqlalchemy import create_engine # 2.0.48
from sqlalchemy.types import NVARCHAR, VARCHAR, TEXT, INT

In [30]:
def bulk_insert_sql(df,server, database, driver):
    connection_string = f'mssql+pyodbc://@{server}/{database}?driver={driver}&Trusted_Connection=yes&TrustServerCertificate=yes'
    engine = create_engine(connection_string)

    # Use unicode format
    sql_data_types = {
        'name': NVARCHAR(length=255),
        'description': TEXT(),
        'phone': VARCHAR(length=50),
        'email': VARCHAR(length=255),
        'website': VARCHAR(length=255),
        'address_line': NVARCHAR(length=255),
        'city': NVARCHAR(length=100),
        'capacity': INT()
    }


    df.to_sql(
        'venues', 
        con=engine, 
        if_exists='append', 
        index=False,
        dtype=sql_data_types
    )

    print("Success")

In [ ]:
server = 'SERVER_NAME' # Replace with your server name
database = 'DATABASE_NAME' # Replace with your database name
driver = 'ODBC Driver 18 for SQL Server' # Replace with your driver

bulk_insert_sql(pd.DataFrame(final), server, database, driver)

Success
